<a href="https://colab.research.google.com/github/tukamilano/Semi-Thue/blob/main/Levenshtein_%E5%AD%A6%E7%BF%92%E7%94%A8%E3%83%87%E3%83%BC%E3%82%BF%E3%81%AE%E5%8F%8E%E9%9B%86_%E4%B8%A6%E5%88%97%E5%8C%96.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

search_modeはgbfs(模倣学習に用いるデータ)

In [2]:
!pip -q install python-Levenshtein pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 42.3 MB/s eta 0:00:00


In [6]:
import multiprocessing as mp
from functools import partial
import json, random, os, time
from Levenshtein import distance as levenshtein_distance
import heapq
from typing import List, Tuple, Dict, Optional
from tqdm import tqdm

# =========================
# ルール: 保存/読み込み
# =========================
def make_alphabet(n: int) -> Tuple[str, ...]:
    return tuple(chr(65 + (i % 26)) for i in range(n))

def sample_geometric_len(p: float) -> int:
    i = 0
    while True:
        if random.random() < p:
            return i
        i += 1

def generate_rules(alphabet: Tuple[str, ...], rule_num: int, p: float) -> List[Tuple[str, str]]:
    rules = set()
    while len(rules) < rule_num:
        i = sample_geometric_len(p); j = sample_geometric_len(p)
        before = ''.join(random.choices(alphabet, k=i))
        after  = ''.join(random.choices(alphabet, k=j))
        if before == after:
            continue
        rules.add((before, after))
    # リスト化＆ソートして順序を固定（ピクル/マルチプロセスでも安定）
    return sorted(list(rules))

def save_rules(path: str, rules: List[Tuple[str, str]], *, alphabet_num: int, rule_num: int, p: float, base_seed_rules: int):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    payload = {
        "config": {"alphabet_num": alphabet_num, "rule_num": rule_num, "p": p},
        "base_seed_rules": base_seed_rules,
        # JSONの都合でタプルは配列として保存
        "rules": rules
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)

def load_rules(path: str) -> Dict:
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    # 読み込んだらタプル化（任意）
    obj["rules"] = [tuple(x) for x in obj["rules"]]
    return obj

def prepare_rules(rules_path: str, *, alphabet_num: int, rule_num: int, p: float, base_seed_rules: int):
    """
    rules.json があれば読み込み、無ければ base_seed_rules を使って生成して保存。
    返り値: (alphabet, rules, meta_dict)
    """
    if os.path.exists(rules_path):
        meta = load_rules(rules_path)
        alphabet = make_alphabet(meta["config"]["alphabet_num"])
        rules = meta["rules"]
        return alphabet, rules, meta
    else:
        # ルール生成の再現性を担保
        rng = random.Random(base_seed_rules)
        random.setstate(rng.getstate())
        alphabet = make_alphabet(alphabet_num)
        rules = generate_rules(alphabet, rule_num, p)
        save_rules(rules_path, rules, alphabet_num=alphabet_num, rule_num=rule_num, p=p, base_seed_rules=base_seed_rules)
        meta = {"config": {"alphabet_num": alphabet_num, "rule_num": rule_num, "p": p},
                "base_seed_rules": base_seed_rules}
        return alphabet, rules, meta

# =========================
# 探索本体（rule_idxも保持）
# =========================
def gbfs_trajectory(before: str, after: str, rules: List[Tuple[str, str]], threshold: int = 100000) -> Optional[List[Dict]]:
    """
    Greedy Best-First Search guided by Levenshtein distance.
    Returns a list of steps like:
      {"state": str, "rule": [lhs, rhs], "rule_idx": int, "pos": int, "next_state": str}
    - pos: ルール適用の開始インデックス（lhs=="" の挿入ルールでは挿入位置 0..len(state)）
    """
    if before == after:
        return []

    pq = [(levenshtein_distance(before, after), 0, before)]
    visited = {before}
    # child_state -> (parent_state, (lhs, rhs), pos, rule_idx)
    parent: Dict[str, Tuple[str, Tuple[str, str], int, int]] = {}

    expanded = 0
    rule_to_idx = {r: i for i, r in enumerate(rules)}

    while pq:
        h, depth, cur = heapq.heappop(pq)

        for lhs, rhs in rules:
            blen = len(lhs)
            if blen == 0:
                positions = range(len(cur) + 1)  # 挿入: 0..len(cur)
            else:
                if len(cur) < blen:
                    continue
                positions = range(len(cur) - blen + 1)

            for pos in positions:
                if cur[pos:pos+blen] == lhs:
                    new_state = cur[:pos] + rhs + cur[pos+blen:]
                    if new_state in visited:
                        continue
                    visited.add(new_state)

                    r_idx = rule_to_idx[(lhs, rhs)]
                    parent[new_state] = (cur, (lhs, rhs), pos, r_idx)
                    expanded += 1

                    if new_state == after:
                        # 復元
                        nodes = [after]
                        while nodes[-1] != before:
                            prev, _, _, _ = parent[nodes[-1]]
                            nodes.append(prev)
                        nodes.reverse()

                        steps: List[Dict] = []
                        for s, t in zip(nodes[:-1], nodes[1:]):
                            _, (plhs, prhs), ppos, pr_idx = parent[t]
                            steps.append({
                                "state": s,
                                "rule": [plhs, prhs],
                                "rule_idx": pr_idx,
                                "pos": ppos,
                                "next_state": t
                            })
                        return steps

                    heapq.heappush(pq, (levenshtein_distance(new_state, after), depth + 1, new_state))

                    if expanded >= threshold:
                        return None

    return None

def one_trial(idx: int, alphabet: Tuple[str, ...], p: float, rules: List[Tuple[str, str]], threshold: int, base_seed: int):
    # 再現性: 各試行 seed を base_seed から決定
    random.seed((base_seed + 2654435761 * (idx + 1)) & 0x7fffffff)
    i = sample_geometric_len(p); j = sample_geometric_len(p)
    before = ''.join(random.choices(alphabet, k=i))
    after  = ''.join(random.choices(alphabet, k=j))
    traj = gbfs_trajectory(before, after, rules, threshold)
    return {"trial": idx, "before": before, "after": after, "trajectory": traj}

def run_parallel(out_file: str, n_trials: int, procs: int, alphabet, p, rules,
                 threshold: int, base_seed: int, chunksize: int = 64):
    """
    並列実行しながら tqdm で進捗を表示
    """
    work = partial(one_trial, alphabet=alphabet, p=p, rules=rules,
                   threshold=threshold, base_seed=base_seed)

    with mp.Pool(processes=procs) as pool, open(out_file, "w", encoding="utf-8") as f:
        # tqdm で進捗バー＋残り時間推定
        for rec in tqdm(pool.imap_unordered(work, range(n_trials), chunksize=chunksize),
                        total=n_trials, desc="Trials"):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# =========================
# 実行例
# =========================
if __name__ == "__main__":
    # 重要: ルールを別ファイルで固定して使い回す
    RULES_PATH = "bfs_experiments/rules.json"
    os.makedirs(os.path.dirname(RULES_PATH), exist_ok=True)

    # 学習で今後も使うルール：固定シードで生成＆保存／なければ読み込み
    # ※ ここを変えない限り、将来の実験でも同じ rules.json が使われます
    alphabet_num, rule_num, p = 25, 500, 0.2
    base_seed_rules = 123456789   # ← ルール生成専用シード（重要！変えない）
    alphabet, rules, meta = prepare_rules(
        RULES_PATH,
        alphabet_num=alphabet_num, rule_num=rule_num, p=p,
        base_seed_rules=base_seed_rules
    )
    print("Loaded rules:", RULES_PATH, "config=", meta["config"], "seed=", meta["base_seed_rules"])

    # データ生成
    out_file = "test_parallel.jsonl"
    n_trials = 1000000
    threshold = 100000

    # 試行のシード（ルールとは独立）：変えてもルールは不変
    base_seed_trials = 424242

    # Colab：2並列 / 本番：os.cpu_count() など
    procs = os.cpu_count()
    run_parallel(out_file, n_trials, procs, alphabet, p, rules, threshold, base_seed=base_seed_trials)
    print("wrote:", out_file)



Loaded rules: /content/bfs_experiments/rules.json config= {'alphabet_num': 25, 'rule_num': 500, 'p': 0.2} seed= 123456789
wrote: /content/test_parallel.jsonl


In [7]:
import os

file_path = "/content/test_parallel.jsonl"
file_size_bytes = os.path.getsize(file_path)
print(f"The size of the file {file_path} is: {file_size_bytes} bytes")

The size of the file /content/test_parallel.jsonl is: 505686 bytes
